for:
one-hot encoding of genres  
feature-matris  
experiment with cosine similarity  
try KNN

In [30]:
#Interactive library
import plotly.express as px

# Data handling
import numpy as np
import pandas as pd

# Machine learning
import sklearn

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.feature_extraction.text import TfidfVectorizer


In [31]:
directory_data = "../data" # Adjust the path to your data directory if necessary
movies = pd.read_csv(f"{directory_data}/raw/movies.csv")  # Adjust the path to your data directory if necessary
ratings = pd.read_csv(f"{directory_data}/raw/ratings.csv")
tags = pd.read_csv(f"{directory_data}/raw/tags.csv")

This implementation was inspired by a discussion with *Johan Challita* on LinkedIn.
The idea was to start with a simple content-based similarity recommender as a baseline, particularly when considering the influence of highly active users in the dataset.
Based on this discussion, I first implement a similarity-based approach and then introduce a KNN-based model in the next step.

*stopwords* english stop words, you can adjust this for other languages or use a custom list. I know that this will be a problem since the movielens data is not limited to english movies, butr for tags and genres it will still work.

In [38]:
movies["content"] = movies["genres"].str.replace("|", " ", regex=False)
# Create TF-IDF matrix
tfidf = TfidfVectorizer(stop_words="english")   #for english stop words, you can adjust this for other languages or use a custom list
tfidf_matrix = tfidf.fit_transform(movies["content"])

In [39]:

def get_recommendations(movie_title, n=5):
    matches = movies[movies["title"].str.contains(movie_title, case=False, na=False)]

    if len(matches) == 0:
        print(f"Movie could not be found")
        return None
    if len(matches) > 1:
        print("multiple movies matched your search: ")
        print(matches[["title"]].head(10))

    
    movie_index = matches.index[0]
    real_title = movies.loc[movie_index, "title"]

    similarities = cosine_similarity(tfidf_matrix[movie_index], tfidf_matrix).flatten()

    similar_indices = similarities.argsort()[::-1]
    similar_indices = [i for i in similar_indices if i != movie_index]

    result = movies.iloc[similar_indices][["title", "genres"]].head(n).copy()

    print(f"Recommendations for '{real_title}':")

    return result

In [41]:
get_recommendations("incredibles", 5)

multiple movies matched your search: 
                         title
8248   Incredibles, The (2004)
54228     Incredibles 2 (2018)
Recommendations for 'Incredibles, The (2004)':


,title,genres
8639,Asterix and Cleopatra (Astérix et Cléopâtre) (...,Action|Adventure|Animation|Children|Comedy
8623,Asterix and the Gauls (Astérix le Gaulois) (1967),Action|Adventure|Animation|Children|Comedy
61208,Kung Fu Panda: Secrets of the Scroll (2016),Action|Adventure|Animation|Children|Comedy
72425,The Legend of Hallowaiian (2018),Action|Adventure|Animation|Children|Comedy
74713,Wild Kratts: A Creature Christmas (2015),Action|Adventure|Animation|Children|Comedy
